# Exploratory Data Analysis
## RBA Cash Rate and the Household Saving Rate in Australia (2000–2025)

**Research question:** Did reductions in the RBA cash rate reduce the household saving rate in Australia?

**Purpose of this notebook:**  
This EDA pursues three objectives. First, it examines the basic properties of both 
variables and checks that the cleaning and merging steps produced a reliable dataset. 
Second, it explores whether a visible relationship between the cash rate and the saving 
rate exists in the data before any formal modelling. Third, it identifies key features of the data, such as structural breaks, persistence, and potential confounders, which have direct implications for how the regression model should be specified. Each section 
ends with modelling implications drawn from the findings.


---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

print('Setup complete.')

---
## 1. Load and Inspect Cleaned Data

In [ ]:
df = pd.read_csv('../data/clean/final_dataset.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Observations : {len(df)}')
print(f'Period       : {df.date.min().date()} to {df.date.max().date()}')
print(f'\nData types:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum())
df.head(10)

**What was done in cleaning, and does it matter?**

The raw cash rate is published as decision dates only, meaning the RBA records 
a new entry only when the rate changes. To convert this to a quarterly series, 
the announced rate was forward-filled to every calendar day and then averaged 
within each quarter. This approach captures the average policy rate in effect 
throughout each quarter rather than just the end-of-period value.

The saving ratio was not taken directly from a published series but calculated 
from the ABS quarterly household income account as net saving divided by gross 
disposable income, using seasonally adjusted values. This gives 104 quarterly 
observations from 2000Q1 to 2025Q4.


---
## 2. Descriptive Statistics and Variable Characteristics

In [ ]:
desc = df[['saving_rate', 'cash_rate']].describe().T
desc['skewness'] = df[['saving_rate', 'cash_rate']].skew()
desc['kurtosis'] = df[['saving_rate', 'cash_rate']].kurtosis()
print('=== Descriptive Statistics ===')
print(desc.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, var, label, color in zip(
    axes,
    ['saving_rate', 'cash_rate'],
    ['Household Saving Rate (%)', 'RBA Cash Rate (%)'],
    ['steelblue', 'darkorange']
):
    ax.hist(df[var].dropna(), bins=18, color=color, edgecolor='white', linewidth=0.5, density=True, alpha=0.8)
    # Overlay KDE
    kde_x = np.linspace(df[var].min() - 1, df[var].max() + 1, 200)
    kde = stats.gaussian_kde(df[var].dropna())
    ax.plot(kde_x, kde(kde_x), color='black', linewidth=1.5)
    ax.axvline(df[var].mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean = {df[var].mean():.2f}%')
    ax.axvline(df[var].median(), color='navy', linestyle=':', linewidth=1.5,
               label=f'Median = {df[var].median():.2f}%')
    ax.set_title(f'Distribution: {label}', fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../output/fig1_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation and modelling implications:**

- **Saving rate:** Positive skew is expected, driven by the COVID-19 spike in 2020 (Q2 saving ratio reached approximately 20–22% as households could not spend). This is a major outlier relative to the rest of the distribution. OLS is not sensitive to skew in the dependent variable per se, but outliers of this magnitude can have high leverage and bias coefficient estimates. A COVID dummy (or exclusion robustness check) is warranted.

- **Cash rate:** The distribution reflects the post-GFC structural shift: the RBA held rates near historical lows for an extended period, then hiked sharply from 2022. The bimodal or right-skewed shape reflects this regime change. Notably, the cash rate has essentially no variation in 2020–2022 (stuck at 0.1%), which means the COVID spike in saving cannot be attributed to interest rates (a key identification concern).

- **Kurtosis:** Heavy tails in either variable suggest the model errors may also be non-normal, making heteroskedasticity-robust standard errors preferable to conventional OLS standard errors.

---
## 3. Time Series Visualisation

Before testing any relationship, it helps to look at how each series 
has moved over time on its own. This makes it easier to spot trends, 
unusual periods, and whether the two series appear to move together visually.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Saving rate
ax1.plot(df['date'], df['saving_rate'], color='steelblue', linewidth=2)
ax1.fill_between(df['date'], df['saving_rate'], alpha=0.15, color='steelblue')
ax1.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax1.set_ylabel('Saving Rate (%)', fontweight='bold')
ax1.set_title('Household Saving Rate and RBA Cash Rate, Quarterly', fontsize=13, fontweight='bold')

# Cash rate
ax2.plot(df['date'], df['cash_rate'], color='darkorange', linewidth=2)
ax2.fill_between(df['date'], df['cash_rate'], alpha=0.15, color='darkorange')
ax2.set_ylabel('RBA Cash Rate (%)', fontweight='bold')
ax2.set_xlabel('Date')

# Annotate events on both panels
events = [
    ('2011-11-01', 'Rate cut\ncycle begins'),
    ('2020-03-01', 'COVID-19\n& ZLB'),
    ('2022-05-01', 'Rate\nhikes'),
]
for date_str, label in events:
    x = pd.to_datetime(date_str)
    for ax in [ax1, ax2]:
        ax.axvline(x=x, color='grey', linestyle=':', linewidth=1.1, alpha=0.7)
    ax1.text(x, ax1.get_ylim()[1] * 0.9, label, fontsize=8, ha='center',
             color='dimgrey', va='top',
             bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2))

plt.tight_layout()
plt.savefig('../output/fig2_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:**

Three key observations from this plot help us identify the relationship:

First, from 2000 to 2019 both series broadly trend downward together. The cash 
rate fell from around 6% to 0.75% and the saving rate followed a similar 
downward path over the same period. This part of the sample provides the most 
direct visual support for the hypothesis.

Second, the COVID-19 period from 2020 to 2021 stands out clearly as an outlier. 
The saving rate spiked to over 20% while the cash rate was already at its floor 
of 0.10%. This is the opposite of what theory would predict and is explained by 
non-monetary factors such as lockdowns and restricted spending opportunities 
rather than interest rate incentives. Including this period without any controls 
will bias the regression coefficient toward zero or even make it negative.

Third, from 2022 onwards the RBA raised rates sharply from 0.10% to 4.35%. 
If the saving rate also rose over this period it provides additional support 
for the hypothesis in the opposite direction.

A key concern raised by this plot is that the RBA tends to cut rates precisely 
when the economy is already weakening, which is also when households save more 
out of caution. This means a simple regression without controls will underestimate 
the true effect of rate cuts on saving.


---
## 4. Unconditional Correlation and Scatter Analysis

In [ ]:
# Full-sample correlation
r_pearson, p_pearson = stats.pearsonr(df['cash_rate'].dropna(), df['saving_rate'].dropna())
r_spearman, p_spearman = stats.spearmanr(df['cash_rate'].dropna(), df['saving_rate'].dropna())

print('=== Full-sample Correlations ===')
print(f'Pearson r  : {r_pearson:.3f}  (p = {p_pearson:.4f})')
print(f'Spearman r : {r_spearman:.3f}  (p = {p_spearman:.4f})')
print()

# Excluding COVID
df_nocovid = df[~((df['date'] >= '2020-01-01') & (df['date'] <= '2021-12-31'))]
r_nc, p_nc = stats.pearsonr(df_nocovid['cash_rate'], df_nocovid['saving_rate'])
print('=== Excluding COVID-19 Quarters (2020 Q1 – 2021 Q4) ===')
print(f'Pearson r  : {r_nc:.3f}  (p = {p_nc:.4f})')

In [ ]:
# Colour-coded scatter by period
def label_period(d):
    if d < pd.Timestamp('2020-01-01'):
        return 'Pre-COVID (2011–2019)'
    elif d <= pd.Timestamp('2021-12-31'):
        return 'COVID-19 (2020–2021)'
    else:
        return 'Post-COVID / Hikes (2022–2025)'

df['period'] = df['date'].apply(label_period)

palette = {
    'Pre-COVID (2011–2019)': 'steelblue',
    'COVID-19 (2020–2021)': 'red',
    'Post-COVID / Hikes (2022–2025)': 'darkorange'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full sample
for period, color in palette.items():
    sub = df[df['period'] == period]
    axes[0].scatter(sub['cash_rate'], sub['saving_rate'], color=color, alpha=0.7, s=45, label=period)

m, b = np.polyfit(df['cash_rate'].dropna(), df['saving_rate'].dropna(), 1)
x_r = np.linspace(df['cash_rate'].min(), df['cash_rate'].max(), 100)
axes[0].plot(x_r, m*x_r + b, 'k--', linewidth=1.5,
             label=f'OLS (all): slope={m:.2f}, R²={r_pearson**2:.2f}')
axes[0].set_title('Full Sample', fontweight='bold')
axes[0].set_xlabel('Cash Rate (%)')
axes[0].set_ylabel('Saving Rate (%)')
axes[0].legend(fontsize=8)

# Right: excluding COVID
for period, color in palette.items():
    if period == 'COVID-19 (2020–2021)': continue
    sub = df[df['period'] == period]
    axes[1].scatter(sub['cash_rate'], sub['saving_rate'], color=color, alpha=0.7, s=45, label=period)

m2, b2 = np.polyfit(df_nocovid['cash_rate'], df_nocovid['saving_rate'], 1)
axes[1].plot(x_r, m2*x_r + b2, 'k--', linewidth=1.5,
             label=f'OLS (ex-COVID): slope={m2:.2f}, R²={r_nc**2:.2f}')
axes[1].set_title('Excluding COVID-19', fontweight='bold')
axes[1].set_xlabel('Cash Rate (%)')
axes[1].set_ylabel('Saving Rate (%)')
axes[1].legend(fontsize=8)

plt.suptitle('Cash Rate vs Saving Rate: Does Excluding COVID Change the Picture?',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../output/fig3_scatter_covid_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:**

Comparing the two panels directly addresses the most important data quality concern. The COVID-19 quarters are extreme outliers: saving was very high *despite* the cash rate being at its floor. If the full-sample correlation is significantly weaker than the ex-COVID correlation, this demonstrates that the COVID period distorts the bivariate relationship and must be controlled for.

The slope comparison between panels is itself a result worth reporting — it quantifies the extent of COVID-induced bias in a simple regression. This motivates including a COVID dummy variable in all primary specifications.

**On the sign of the slope:** Theory predicts a *positive* slope (higher rate → higher saving; lower rate → lower saving). If the OLS line slopes upward, this is consistent with the hypothesis. If the ex-COVID slope is substantially steeper than the full-sample slope, that is direct evidence that the COVID period is pulling the estimate toward zero.

---
## 5. Lag Structure: When Does the Rate Effect Show Up?

Monetary policy transmission operates with a lag. The literature on the transmission mechanism suggests effects on consumption and saving typically emerge over 2–6 quarters. The EDA should establish *which* lag is most appropriate to use in the primary model.

In [ ]:
# Lag correlation analysis (full sample and ex-COVID)
lag_results = []
for lag in range(0, 9):
    # Full sample
    df_lag = df[['saving_rate', 'cash_rate']].copy()
    df_lag['cash_rate_lag'] = df_lag['cash_rate'].shift(lag)
    df_lag = df_lag.dropna()
    r_full, p_full = stats.pearsonr(df_lag['cash_rate_lag'], df_lag['saving_rate'])
    
    # Ex-COVID
    df_lag_nc = df_nocovid[['saving_rate', 'cash_rate']].copy()
    df_lag_nc['cash_rate_lag'] = df_nocovid['cash_rate'].shift(lag).values[:len(df_nocovid)]
    df_lag_nc = df_lag_nc.dropna()
    if len(df_lag_nc) > 5:
        r_nc_lag, p_nc_lag = stats.pearsonr(df_lag_nc['cash_rate_lag'], df_lag_nc['saving_rate'])
    else:
        r_nc_lag, p_nc_lag = np.nan, np.nan
    
    lag_results.append({'Lag (quarters)': lag, 'r (full)': r_full, 'p (full)': p_full,
                        'r (ex-COVID)': r_nc_lag, 'p (ex-COVID)': p_nc_lag})

lag_df = pd.DataFrame(lag_results)
print('=== Lag Correlation Table ===')
print(lag_df.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = lag_df['Lag (quarters)']
width = 0.35

bars1 = ax.bar(x - width/2, lag_df['r (full)'], width, label='Full sample',
               color='steelblue', alpha=0.8, edgecolor='white')
bars2 = ax.bar(x + width/2, lag_df['r (ex-COVID)'], width, label='Excl. COVID-19',
               color='darkorange', alpha=0.8, edgecolor='white')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Lag (quarters)', fontweight='bold')
ax.set_ylabel('Pearson Correlation with Saving Rate', fontweight='bold')
ax.set_title('Lag Correlations: Cash Rate and Saving Rate\n(Full Sample vs Excluding COVID-19)',
             fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Lag {l}' for l in x])
ax.legend()

plt.tight_layout()
plt.savefig('../output/fig4_lag_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation and modelling implications:**

This plot serves as the primary input to the lag selection decision for the primary model. Several observations to look for:

- If correlations **peak at Lag 1–4** (i.e., 1–4 quarters), this confirms a transmission lag and the model should include the lagged rather than contemporaneous cash rate.
- If the **ex-COVID correlation profile** is smoother and stronger than the full-sample one, this further confirms COVID as a confounding period.
- If correlations are uniformly low even after excluding COVID, this suggests the bivariate relationship is genuinely weak and multivariate controls (income growth, unemployment, consumer confidence) are needed before drawing any conclusion.

**Note on interpretation:** High correlation at a particular lag does not establish causality — the RBA's rate decisions are forward-looking and correlated with the business cycle. The formal model needs instruments or controls for this endogeneity, which will be discussed in the primary analysis.

---
## 6. Stationarity Testing

Running OLS on non-stationary time series can produce spurious results — high R² and significant t-statistics even when no true relationship exists. This section tests the order of integration of both series, which directly determines the appropriate regression specification.

In [ ]:
def stationarity_report(series, name, maxlag=8):
    s = series.dropna()
    
    # ADF test
    adf_stat, adf_p, _, _, adf_cv, _ = adfuller(s, maxlags=maxlag, autolag='AIC', regression='c')
    adf_with_trend = adfuller(s, maxlags=maxlag, autolag='AIC', regression='ct')
    
    print(f'--- {name} ---')
    print(f'  ADF (const):       stat={adf_stat:.3f}, p={adf_p:.4f}, 5% crit={adf_cv["5%"]:.3f}')
    print(f'  ADF (const+trend): stat={adf_with_trend[0]:.3f}, p={adf_with_trend[1]:.4f}')
    
    # First difference
    ds = s.diff().dropna()
    adf_d, adf_dp, _, _, adf_dcv, _ = adfuller(ds, maxlags=maxlag, autolag='AIC', regression='c')
    print(f'  ADF (Δ, const):    stat={adf_d:.3f}, p={adf_dp:.4f}, 5% crit={adf_dcv["5%"]:.3f}')
    
    if adf_p > 0.05:
        verdict = 'Non-stationary in levels'
        if adf_dp < 0.05:
            verdict += ' → I(1): stationary in first differences'
        else:
            verdict += ' → Possibly I(2) or structurally broken'
    else:
        verdict = 'Stationary in levels → I(0)'
    print(f'  Verdict: {verdict}')
    print()

print('=== Augmented Dickey-Fuller Unit Root Tests ===')
stationarity_report(df['saving_rate'], 'Saving Rate')
stationarity_report(df['cash_rate'], 'Cash Rate')

**Interpretation and modelling implications:**

This is one of the most important outputs of the EDA, as the result directly 
determines the regression specification. If both variables are stationary in levels, 
OLS in levels is valid and standard inference applies. If both are non-stationary 
and cointegrated, a levels regression remains appropriate since a long-run equilibrium 
relationship exists, which can be tested using the Engle-Granger or Johansen approach. 
If both are non-stationary but not cointegrated, both series must be first-differenced, 
and the regression captures the short-run effect of quarterly changes in the cash rate 
on quarterly changes in the saving rate. Mixed integration orders require an ARDL 
bounds testing approach.

Given the strong downward trends in both series over 2000-2019, neither is likely 
stationary in levels. First-differencing is therefore the conservative and defensible 
baseline. The interpretation then shifts from "does a higher cash rate lead to higher 
saving?" to "does a cut in the cash rate lead to a decrease in the saving rate in the 
same or following quarter?"

ADF tests have low power in short samples. With around 100 quarterly observations 
over 2000-2025, findings should be treated as indicative rather than definitive, and 
results are best interpreted alongside the visual evidence from the time series plots.

---
## 7. Autocorrelation in the Saving Rate

Understanding the autocorrelation structure of the dependent variable is important for choosing the right estimator and standard errors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_acf(df['saving_rate'].dropna(), lags=12, ax=axes[0], title='ACF: Saving Rate (levels)')
plot_acf(df['saving_rate'].diff().dropna(), lags=12, ax=axes[1],
         title='ACF: Saving Rate (first differences)')

plt.tight_layout()
plt.savefig('../output/fig5_acf.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation and modelling implications:**

- **Strong autocorrelation in levels** (many significant spikes in the ACF) confirms non-stationarity and persistence. Households do not adjust saving rates instantaneously.
- **If first-differenced series still shows autocorrelation** at lags 1–2, this suggests the primary regression should include a lagged dependent variable (Δsaving_rate(t-1)) to capture persistence in saving changes. This is standard in dynamic models of saving behaviour.
- Residual autocorrelation from a mis-specified model violates OLS assumptions and inflates t-statistics. Using Newey-West (HAC) standard errors is recommended as a robustness check.

---
## 8. Potential Confounders: Variable Correlation Matrix

For the primary model to isolate the effect of the cash rate on saving, we need to understand which other variables are correlated with both the cash rate and the saving rate (omitted variable bias).

In [ ]:
# Note: if you add control variables (unemployment rate, CPI inflation, etc.)
# to the clean dataset, include them here. Otherwise this section provides
# a template and discusses the expected relationships.

print('=== Potential confounders and their expected relationship with cash rate and saving rate ===')
print()
confounders = {
    'Unemployment rate': (
        'Positive with saving (precautionary motive)',
        'Negative with cash rate (RBA cuts when unemployment rises)',
        'Omitting this creates downward bias: cutting rates happens when unemployment rises → saving rises'
    ),
    'CPI inflation': (
        'Negative with saving (real returns on deposits fall)',
        'Positive with cash rate (RBA hikes to fight inflation)',
        'Direction of bias is ambiguous'
    ),
    'Household income growth': (
        'Ambiguous with saving (higher income → more saving in absolute terms, but saving ratio may fall if consumption rises faster)',
        'Positive with cash rate (strong economy → less need to cut)',
        'Omitting this likely causes upward bias'
    ),
    'Housing price growth': (
        'Negative with saving (wealth effect reduces need to save)',
        'Negative with cash rate (low rates → house price boom)',
        'Omitting this could create downward bias'
    ),
}

for name, (saving_rel, rate_rel, bias) in confounders.items():
    print(f'{name}')
    print(f'  Relationship with saving rate: {saving_rel}')
    print(f'  Relationship with cash rate:   {rate_rel}')
    print(f'  Bias if omitted:               {bias}')
    print()

**Modelling implications:**

The most important confounder is unemployment, or more broadly, economic uncertainty. 
The RBA lowers rates precisely when the economy weakens, and households also tend to 
save more precautionarily during the same periods. Without controlling for this, the 
OLS coefficient on the cash rate will be biased toward zero, meaning the true effect 
of rate cuts on saving is likely larger than a naive estimate would suggest. This is 
a well-known problem in the monetary policy transmission literature.

Even a simple control for the unemployment rate significantly reduces this bias. 
The ABS Labour Force Survey provides monthly unemployment data that is easily 
converted to quarterly averages. Including this variable does not change the research 
question. It makes the answer to it more credible.


In [ ]:
print("""
EDA SUMMARY AND PRIMARY MODELLING ROADMAP

Data
- N = 104 quarterly observations, 2000 to 2025
- No missing values after cleaning
- Cash rate: quarterly average of daily forward-filled RBA target rate
- Saving ratio: ABS 5206.0 Table 34, seasonally adjusted

Finding 1 - COVID is a structural break
- Saving spiked in 2020-2021 while rates were at zero
- Bivariate correlation weakens or flips when COVID quarters are included
- Include a COVID dummy variable in all specifications

Finding 2 - First-order effect exists
- Positive correlation (excluding COVID) supports the hypothesis
- R² is low, meaning other factors also matter
- Add unemployment rate as a control variable

Finding 3 - Lag structure
- Correlation is strongest at Lag 0(contemporaneous)
- The cash rate and saving rate move together within the same quarter

Finding 4 - Stationarity
- Both series appear non-stationary in levels (I(1))
- Use first differences as the baseline specification
- Test for cointegration as a robustness check

Finding 5 - Autocorrelation
- Saving rate is persistent
- Include lagged dependent variable in the model
- Use Newey-West HAC standard errors

Proposed baseline model:
  change_saving = a + b*change_cash_rate(t-k) + c*change_unemp + d*change_saving(t-1) + e*COVID + error
""")